## Google Colab Setup

Mount Google Drive, pull data + utility script from GitHub, then train.
The setup cell also downloads a fresh ESM-2 snapshot from Hugging Face on every run so the notebook never reuses a stale local cache.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ── Colab output directories on Drive ────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/XAllergen2.0')
DRIVE_MODELS = DRIVE_ROOT / 'models'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Drive directories ready:')
print(f'  {DRIVE_MODELS}')
print(f'  {DRIVE_RESULTS}')


Mounted at /content/drive
Drive directories ready:
  /content/drive/MyDrive/XAllergen2.0/models
  /content/drive/MyDrive/XAllergen2.0/results


In [2]:
import os
import urllib.request
from pathlib import Path

from huggingface_hub import snapshot_download

RUNTIME_ROOT = Path('/content/XAllergen2.0')
SRC_DIR = RUNTIME_ROOT / 'src'
PACKAGE_DIR = SRC_DIR / 'xallergen'
DATA_DIR   = RUNTIME_ROOT / 'data'
MODEL_DIR  = RUNTIME_ROOT / 'models'
RESULTS_DIR = RUNTIME_ROOT / 'results'
FRESH_ESM2_DIR = RUNTIME_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'
for d in [DATA_DIR, MODEL_DIR, RESULTS_DIR, FRESH_ESM2_DIR, PACKAGE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW = 'https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main'

# Package files
urllib.request.urlretrieve(f'{RAW}/src/xallergen/__init__.py', PACKAGE_DIR / '__init__.py')
urllib.request.urlretrieve(
    f'{RAW}/src/xallergen/baseline_notebook_utils.py',
    PACKAGE_DIR / 'baseline_notebook_utils.py',
)

# Always download a fresh ESM-2 snapshot and force downstream code to use it
FRESH_ESM2_PATH = snapshot_download(
    repo_id='facebook/esm2_t6_8M_UR50D',
    local_dir=FRESH_ESM2_DIR,
    local_dir_use_symlinks=False,
    force_download=True,
    resume_download=False,
)
os.environ['XALLERGEN_HF_MODEL_DIR'] = str(FRESH_ESM2_PATH)

# Data files
DATA_FILES = [
    'deepalgpro_test_cleaned.csv',
    'deepalgpro_train_cleaned.csv',
    'negatives_splitA.csv',
    'negatives_splitB.csv',
    'positives_splitA.csv',
    'positives_splitB.csv',
]
for fname in DATA_FILES:
    urllib.request.urlretrieve(f'{RAW}/data/{fname}', DATA_DIR / fname)

print('Downloaded:')
print('  Utility script: src/xallergen/baseline_notebook_utils.py')
print(f'  Fresh ESM-2 snapshot: {FRESH_ESM2_PATH}')
for p in sorted(RUNTIME_ROOT.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(RUNTIME_ROOT)}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded:
  Utility script: src/xallergen/baseline_notebook_utils.py
  Fresh ESM-2 snapshot: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
  data/deepalgpro_test_cleaned.csv
  data/deepalgpro_train_cleaned.csv
  data/negatives_splitA.csv
  data/negatives_splitB.csv
  data/positives_splitA.csv
  data/positives_splitB.csv
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/.gitignore
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/CACHEDIR.TAG
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/.gitattributes.metadata
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/README.md.metadata
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/config.json.metadata
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/model.safetensors.metadata
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/pytorch_model.bin.metadata
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/special_toke

# Top-1-unfrozen ESM-2 baseline for DeepAlgPro

This notebook trains a top-1-unfrozen `esm2_t6_8M_UR50D` allergenicity classifier on `data/deepalgpro_train_cleaned.csv` and evaluates it on the held-out `data/deepalgpro_test_cleaned.csv`.

Architecture:
- Backbone: `esm2_t6_8M_UR50D`, with only the final transformer layer unfrozen during training.
- Attention pooling: `Linear(embed_dim -> 1)` per residue, softmax-normalized across sequence length, then a weighted sum of residue embeddings. The resulting per-residue weights are kept as an attribution signal.
- Classification head: `Linear(embed_dim -> 128) -> ReLU -> Dropout(0.3) -> Linear(128 -> 1)`.
- Training loss: class-weighted `BCEWithLogitsLoss`, matching the frozen baseline; inference uses `sigmoid`.

Split strategy:
- `deepalgpro_train_cleaned.csv` is split into train and validation with a stratified random 90/10 split.
- `sklearn.model_selection.train_test_split` is used with `stratify=df["label"]` and `random_state=42`.
- `deepalgpro_test_cleaned.csv` remains fully held out until final evaluation.


In [3]:
import json
import math
import sys
import time as _time
from pathlib import Path

# Allow imports from the src-layout package without a root-level shim.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _candidate / "src"
    if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
        break

# Add runtime src directory to path so the xallergen package is importable
RUNTIME_ROOT = Path('/content/XAllergen2.0')
SRC_DIR = RUNTIME_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from xallergen.baseline_notebook_utils import (
    DROPOUT,
    ESM_MODEL_NAME,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    RANDOM_STATE,
    THRESHOLD,
    FrozenESMAllergenClassifier,
    build_tokenizer,
    compute_attention_weights,
    compute_integrated_gradients,
    load_baseline_checkpoint,
    normalize_scores,
    configure_backbone_trainability,
    assert_backbone_trainability_mode,
    seed_everything,
)

import matplotlib
matplotlib.use('Agg')  # headless-safe on Colab
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# ── Hyperparameters ───────────────────────────────────────────────────────
BATCH_SIZE    = 24
EPOCHS        = 30
PATIENCE      = 5
MIN_DELTA     = 1e-3
LEARNING_RATE = 4e-6
WEIGHT_DECAY  = 1e-4

# ── Runtime paths ─────────────────────────────────────────────────────────
DATA_DIR    = RUNTIME_ROOT / 'data'
MODEL_DIR   = RUNTIME_ROOT / 'models'
RESULTS_DIR = RUNTIME_ROOT / 'results'

TRAIN_CSV       = DATA_DIR / 'deepalgpro_train_cleaned.csv'
TEST_CSV        = DATA_DIR / 'deepalgpro_test_cleaned.csv'
CHECKPOINT_PATH = MODEL_DIR / 'baseline_top1_unfrozen_esm2.pt'
METRICS_PATH    = RESULTS_DIR / 'baseline_top1_unfrozen_esm2_metrics.json'
BACKBONE_TRAIN_MODE = 'top1_unfrozen'

# ── Drive output paths (set in cell 0) ───────────────────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive/XAllergen2.0')
DRIVE_MODELS  = DRIVE_ROOT / 'models'
DRIVE_RESULTS = DRIVE_ROOT / 'results'

seed_everything(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


In [4]:
train_df = pd.read_csv(TRAIN_CSV).copy()
test_df = pd.read_csv(TEST_CSV).copy()
for df in [train_df, test_df]:
    df["sequence_id"] = df["sequence_id"].astype(str)
    df["sequence"] = df["sequence"].astype(str).str.strip().str.upper()
    df["label"] = df["label"].astype(int)

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df["label"],
)
train_split_df = train_split_df.reset_index(drop=True).copy()
val_split_df = val_split_df.reset_index(drop=True).copy()

print(f"Train sequences: {len(train_split_df):,}")
print(train_split_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())
print()
print(f"Validation sequences: {len(val_split_df):,}")
print(val_split_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())
print()
print(f"Held-out test sequences: {len(test_df):,}")
print(test_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())

Train sequences: 4,995
label
negative    2545
positive    2450

Validation sequences: 556
label
negative    283
positive    273

Held-out test sequences: 1,377
label
negative    705
positive    672


In [5]:
class SequenceDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        return {
            "sequence_id": row["sequence_id"],
            "sequence": row["sequence"],
            "label": int(row["label"]),
        }


tokenizer = build_tokenizer(HF_MODEL_NAME)


def collate_batch(batch: list[dict]) -> dict:
    sequences = [item["sequence"] for item in batch]
    encodings = tokenizer(
        sequences,
        add_special_tokens=False,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )
    return {
        "sequence_id": [item["sequence_id"] for item in batch],
        "sequence": sequences,
        "label": torch.tensor([item["label"] for item in batch], dtype=torch.float32),
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
    }


def move_batch_to_device(batch: dict, device: str) -> dict:
    moved = dict(batch)
    moved["input_ids"] = batch["input_ids"].to(device)
    moved["attention_mask"] = batch["attention_mask"].to(device)
    moved["label"] = batch["label"].to(device)
    return moved


train_loader = DataLoader(SequenceDataset(train_split_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_batch)
val_loader = DataLoader(SequenceDataset(val_split_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_batch)
test_loader = DataLoader(SequenceDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_batch)

model = FrozenESMAllergenClassifier(HF_MODEL_NAME, hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
trainability_summary = configure_backbone_trainability(model, BACKBONE_TRAIN_MODE)
trainability_assertion = assert_backbone_trainability_mode(model, BACKBONE_TRAIN_MODE)
assert trainability_summary['final_block_path'] == trainability_assertion['final_block_path']

trainable_params = [param for param in model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
_counts = train_split_df["label"].value_counts()
_n_neg = int(_counts.get(0, 0))
_n_pos = int(_counts.get(1, 0))
protein_pos_weight = torch.tensor([_n_neg / _n_pos], dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=protein_pos_weight)
print("Protein-level class balance:")
print(f"  negatives: {_n_neg:,}")
print(f"  positives: {_n_pos:,}")
print(f"  pos_weight: {protein_pos_weight.item():.4f}")

print(f"Trainable parameter tensors: {len(trainable_params)}")
print(f"Backbone hidden size: {model.backbone.config.hidden_size}")
print(f"Trainable final block: {trainability_summary['final_block_path']}")


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Backbone trainability summary:
  mode: top1_unfrozen
  total backbone params: 7,511,801
  trainable backbone params: 1,232,960
  percent trainable backbone: 16.41%
  trainable backbone submodules:
    model.backbone.encoder.layer.5
  detected final encoder block: model.backbone.encoder.layer.5
Protein-level class balance:
  negatives: 2,545
  positives: 2,450
  pos_weight: 1.0388
Trainable parameter tensors: 22
Backbone hidden size: 320
Trainable final block: model.backbone.encoder.layer.5


In [6]:
from typing import Optional, Tuple
from tqdm.auto import tqdm

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: str,
    epoch: int,
) -> float:
    model.train()
    total_loss = 0.0
    total_examples = 0
    for batch in loader:  # ← plain loop, no inner tqdm
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        loss = criterion(outputs["logits"], batch["label"])
        loss.backward()
        optimizer.step()
        batch_size = batch["label"].shape[0]
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size
    return total_loss / max(total_examples, 1)


@torch.no_grad()
def predict(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    criterion: Optional[nn.Module] = None,
    desc: Optional[str] = None,
) -> Tuple[Optional[float], pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_examples = 0
    rows = []
    for batch in loader:  # ← plain loop, no inner tqdm
        batch = move_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        logits = outputs["logits"]
        probs = torch.sigmoid(logits)
        if criterion is not None:
            loss = criterion(logits, batch["label"])
            batch_size = batch["label"].shape[0]
            total_loss += float(loss.item()) * batch_size
            total_examples += batch_size
        for idx in range(len(batch["sequence_id"])):
            rows.append(
                {
                    "sequence_id": batch["sequence_id"][idx],
                    "sequence": batch["sequence"][idx],
                    "label": int(batch["label"][idx].item()),
                    "logit": float(logits[idx].item()),
                    "pred_prob": float(probs[idx].item()),
                    "pred_label": int(probs[idx].item() >= THRESHOLD),
                }
            )
    loss_value = None if criterion is None else total_loss / max(total_examples, 1)
    return loss_value, pd.DataFrame(rows)


def compute_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label"].to_numpy()
    y_prob = pred_df["pred_prob"].to_numpy()
    y_pred = pred_df["pred_label"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        "threshold": THRESHOLD,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "auroc": float(roc_auc_score(y_true, y_prob)),
        "auprc": float(average_precision_score(y_true, y_prob)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else math.nan,
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }
    return metrics


history = []
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

_epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Training", unit="epoch", file=sys.stdout, dynamic_ncols=True)
for epoch in _epoch_bar:
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, epoch)
    val_loss, val_pred_df = predict(
        model, val_loader, DEVICE, criterion,
    )
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "esm_model_name": ESM_MODEL_NAME,
                "architecture_hyperparameters": {
                    "hidden_dim": HIDDEN_DIM,
                    "dropout": DROPOUT,
                },
                "training_history": history,
                "backbone_train_mode": BACKBONE_TRAIN_MODE,
                "backbone_trainability": trainability_summary,
            },
            CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    # ← print each epoch result as a plain line so you always see progress
    print(
        f"Epoch {epoch:>3}/{EPOCHS} | "
        f"train_loss={train_loss:.5f} | "
        f"val_loss={val_loss:.5f} | "
        f"best={best_epoch}"
    )
    _epoch_bar.set_postfix(
        train_loss=f"{train_loss:.5f}",
        val_loss=f"{val_loss:.5f}",
        best=best_epoch,
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print(f"Best validation loss: {best_val_loss:.5f} at epoch {best_epoch}")
print(f"Checkpoint saved to: {CHECKPOINT_PATH}")

Training:   0%|          | 0/30 [00:00<?, ?epoch/s]

Epoch   1/30 | train_loss=0.70717 | val_loss=0.70598 | best=1
Epoch   2/30 | train_loss=0.70358 | val_loss=0.70202 | best=2
Epoch   3/30 | train_loss=0.69944 | val_loss=0.69793 | best=3
Epoch   4/30 | train_loss=0.69478 | val_loss=0.69386 | best=4
Epoch   5/30 | train_loss=0.69060 | val_loss=0.68978 | best=5
Epoch   6/30 | train_loss=0.68641 | val_loss=0.68565 | best=6
Epoch   7/30 | train_loss=0.68192 | val_loss=0.68140 | best=7
Epoch   8/30 | train_loss=0.67826 | val_loss=0.67706 | best=8
Epoch   9/30 | train_loss=0.67303 | val_loss=0.67271 | best=9
Epoch  10/30 | train_loss=0.66856 | val_loss=0.66821 | best=10
Epoch  11/30 | train_loss=0.66336 | val_loss=0.66355 | best=11
Epoch  12/30 | train_loss=0.65836 | val_loss=0.65880 | best=12
Epoch  13/30 | train_loss=0.65414 | val_loss=0.65396 | best=13
Epoch  14/30 | train_loss=0.64922 | val_loss=0.64905 | best=14
Epoch  15/30 | train_loss=0.64422 | val_loss=0.64420 | best=15
Epoch  16/30 | train_loss=0.63952 | val_loss=0.63931 | best=16
E

In [7]:
model, checkpoint = load_baseline_checkpoint(
    CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
)

train_history_df = pd.DataFrame(checkpoint["training_history"])

val_loss, val_predictions_df = predict(model, val_loader, DEVICE, criterion)
test_loss, test_predictions_df = predict(model, test_loader, DEVICE, criterion)

test_metrics = compute_metrics(test_predictions_df)
test_metrics["test_loss"] = float(test_loss)
test_metrics["best_epoch"] = int(best_epoch)
test_metrics["n_test_sequences"] = int(len(test_predictions_df))

metrics_payload = {
    "model_family": "baseline",
    "model_variant": BACKBONE_TRAIN_MODE,
    "esm_model_name": ESM_MODEL_NAME,
    "architecture_hyperparameters": {"hidden_dim": HIDDEN_DIM, "dropout": DROPOUT},
    "training": {
        "batch_size": BATCH_SIZE,
        "epochs_requested": EPOCHS,
        "early_stopping_patience": PATIENCE,
        "min_delta": MIN_DELTA,
        "optimizer": "AdamW",
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "protein_pos_weight": protein_pos_weight.item(),
        "protein_pos_weight_formula": "n_neg / n_pos in train split after 90/10 stratified split",
        "backbone_train_mode": BACKBONE_TRAIN_MODE,
        "backbone_trainability": checkpoint.get("backbone_trainability", trainability_summary),
    },
    "validation_loss": float(val_loss),
    "test_metrics": test_metrics,
}

with METRICS_PATH.open("w") as handle:
    json.dump(metrics_payload, handle, indent=2)


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
print('Runtime outputs:')
print(f'  Checkpoint : {CHECKPOINT_PATH}')
print(f'  Metrics    : {METRICS_PATH}')
print(f'  (Will be copied to Drive in the final cell.)')


Runtime outputs:
  Checkpoint : /content/XAllergen2.0/models/baseline_top1_unfrozen_esm2.pt
  Metrics    : /content/XAllergen2.0/results/baseline_top1_unfrozen_esm2_metrics.json
  (Will be copied to Drive in the final cell.)


In [9]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_history_df["epoch"], train_history_df["train_loss"], label="Train loss")
axes[0].plot(train_history_df["epoch"], train_history_df["val_loss"], label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

cm = confusion_matrix(test_predictions_df["label"], test_predictions_df["pred_label"], labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1])
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")
axes[1].set_xticklabels(["Negative", "Positive"])
axes[1].set_yticklabels(["Negative", "Positive"], rotation=0)

plt.tight_layout()
plt.show()

display(Markdown("## Test metrics"))
display(pd.DataFrame([{k: v for k, v in test_metrics.items() if k != 'confusion_matrix'}]).T.rename(columns={0: 'value'}))
print(f"Saved metrics JSON to: {METRICS_PATH}")

## Test metrics

,value
threshold,0.500000
accuracy,0.820625
auroc,0.904310
auprc,0.886050
f1,0.804124
precision,0.860781
recall,0.754464
specificity,0.883688
mcc,0.644737
test_loss,0.568423


Saved metrics JSON to: /content/XAllergen2.0/results/baseline_top1_unfrozen_esm2_metrics.json


In [10]:
def select_attribution_examples(pred_df: pd.DataFrame) -> pd.DataFrame:
    selected_frames = []
    selected_ids = set()

    high_conf_tp = pred_df.loc[
        (pred_df["label"] == 1)
        & (pred_df["pred_label"] == 1)
        & (pred_df["pred_prob"] > 0.9)
    ]
    if len(high_conf_tp) > 0:
        tp_sample = high_conf_tp.sample(n=min(3, len(high_conf_tp)), random_state=RANDOM_STATE)
        selected_frames.append(tp_sample)
        selected_ids.update(tp_sample["sequence_id"])

    errors = pred_df.loc[pred_df["pred_label"] != pred_df["label"]]
    if len(errors) > 0:
        error_sample = errors.sample(n=min(2, len(errors)), random_state=RANDOM_STATE)
        selected_frames.append(error_sample)
        selected_ids.update(error_sample["sequence_id"])

    selected_df = pd.concat(selected_frames, ignore_index=True) if selected_frames else pd.DataFrame(columns=pred_df.columns)

    if len(selected_df) < 5:
        remainder = pred_df.loc[~pred_df["sequence_id"].isin(selected_ids)]
        fill_n = min(5 - len(selected_df), len(remainder))
        if fill_n > 0:
            filler = remainder.sample(n=fill_n, random_state=RANDOM_STATE)
            selected_df = pd.concat([selected_df, filler], ignore_index=True)

    return selected_df.head(5).copy()


def compute_attention_and_ig(sequence: str) -> tuple[np.ndarray, np.ndarray]:
    attention_scores = normalize_scores(
        compute_attention_weights(model, tokenizer, sequence, DEVICE)
    )
    ig_scores = compute_integrated_gradients(
        model,
        tokenizer,
        sequence,
        DEVICE,
        steps=IG_STEPS,
        normalize=True,
    )
    return attention_scores, ig_scores


In [11]:
import shutil

# Copy checkpoint and metrics JSON to Google Drive
shutil.copy(CHECKPOINT_PATH, DRIVE_MODELS / CHECKPOINT_PATH.name)
shutil.copy(METRICS_PATH,    DRIVE_RESULTS / METRICS_PATH.name)

print('Saved to Google Drive:')
print(f'  {DRIVE_MODELS / CHECKPOINT_PATH.name}')
print(f'  {DRIVE_RESULTS / METRICS_PATH.name}')


Saved to Google Drive:
  /content/drive/MyDrive/XAllergen2.0/models/baseline_top1_unfrozen_esm2.pt
  /content/drive/MyDrive/XAllergen2.0/results/baseline_top1_unfrozen_esm2_metrics.json


## Wrap-up

The top-1-unfrozen baseline checkpoint and held-out test metrics have been saved to `/content/XAllergen2.0/` on the Colab runtime **and** copied to your Google Drive under `MyDrive/XAllergen2.0/`. Review the evaluation plots and test metrics above as the controlled top-1-unfrozen baseline reference point.

The next step is dataset-scale attribution evaluation against IEDB epitope ground truth in `03_attribution_evaluation.ipynb`.
